In [1]:
import pandas as pd

In [2]:
d1= pd.read_csv('../data/application_train.csv')
d2= pd.read_csv('../data/bureau.csv')

In [3]:
bureau_features = d2.groupby('SK_ID_CURR').agg(
    TOTAL_PAST_LOANS = ('SK_ID_BUREAU', 'count'),
    TOTAL_PAST_DEFAULTS = ('CREDIT_DAY_OVERDUE', lambda x: (x > 0).sum()),
    AVG_DAYS_OVERDUE = ('CREDIT_DAY_OVERDUE', 'mean'),
    AVG_DEBT_REMAINING = ('AMT_CREDIT_SUM_DEBT', 'mean'),
    TOTAL_CREDIT_SUM = ('AMT_CREDIT_SUM', 'sum')
).reset_index()
 
print("Bureau features shape:", bureau_features.shape)
print(bureau_features.head())

Bureau features shape: (305811, 6)
   SK_ID_CURR  TOTAL_PAST_LOANS  TOTAL_PAST_DEFAULTS  AVG_DAYS_OVERDUE  \
0      100001                 7                    0               0.0   
1      100002                 8                    0               0.0   
2      100003                 4                    0               0.0   
3      100004                 2                    0               0.0   
4      100005                 3                    0               0.0   

   AVG_DEBT_REMAINING  TOTAL_CREDIT_SUM  
0        85240.928571       1453365.000  
1        49156.200000        865055.565  
2            0.000000       1017400.500  
3            0.000000        189037.800  
4       189469.500000        657126.000  


In [4]:
cols = [
    'SK_ID_CURR',
    'TARGET',
    'AMT_INCOME_TOTAL',
    'AMT_CREDIT',
    'AMT_ANNUITY',
    'DAYS_EMPLOYED',
    'DAYS_BIRTH',
    'CNT_CHILDREN',
    'CODE_GENDER',
    'NAME_EDUCATION_TYPE',
    'NAME_FAMILY_STATUS',
    'FLAG_OWN_CAR',
    'FLAG_OWN_REALTY',
    'REGION_POPULATION_RELATIVE',
    'DAYS_REGISTRATION',
    'AMT_GOODS_PRICE',
    'NAME_INCOME_TYPE',
    'ORGANIZATION_TYPE'
]
 
df = d1[cols].copy()
 
# Merge with bureau features
df = df.merge(bureau_features, on='SK_ID_CURR', how='left')
df = df.drop('SK_ID_CURR', axis=1)
 
print("Merged shape:", df.shape)

Merged shape: (307511, 22)


In [5]:
df['DEBT_TO_INCOME'] = df['AMT_CREDIT'] / df['AMT_INCOME_TOTAL']
 
df['ANNUITY_TO_INCOME'] = df['AMT_ANNUITY'] / df['AMT_INCOME_TOTAL']
 
df['CREDIT_TO_GOODS'] = df['AMT_CREDIT'] / df['AMT_GOODS_PRICE']
 
df['AGE_YEARS'] = (-df['DAYS_BIRTH'] / 365).round(1)

df['EMPLOYMENT_YEARS'] = (-df['DAYS_EMPLOYED'] / 365).round(1)
df['DAYS_EMPLOYED'] = df['DAYS_EMPLOYED'].replace(365243, None)
df['EMPLOYMENT_YEARS'] = df['EMPLOYMENT_YEARS'].replace(1000.7, None)
 
df['REGISTRATION_YEARS'] = (-df['DAYS_REGISTRATION'] / 365).round(1)
  
df['EMPLOYMENT_TO_AGE'] = df['EMPLOYMENT_YEARS'] / df['AGE_YEARS']
 
df['CREDIT_TO_AGE'] = df['AMT_CREDIT'] / df['AGE_YEARS']
 
df['INCOME_PER_PERSON'] = df['AMT_INCOME_TOTAL'] / (df['CNT_CHILDREN'] + 1)
 
df['ANNUITY_TO_GOODS'] = df['AMT_ANNUITY'] / df['AMT_GOODS_PRICE']
 
df['LOANS_PER_YEAR'] = df['TOTAL_PAST_LOANS'] / df['AGE_YEARS']
 
df['TOTAL_DEBT_TO_INCOME'] = df['AVG_DEBT_REMAINING'] / df['AMT_INCOME_TOTAL']
df['PAST_DEFAULT_RATE'] = (
    df['TOTAL_PAST_DEFAULTS'] / df['TOTAL_PAST_LOANS']
).fillna(0)

df = df.drop(['DAYS_BIRTH', 'DAYS_EMPLOYED', 'DAYS_REGISTRATION'], axis=1)
 
print("New shape:", df.shape)

New shape: (307511, 32)


In [6]:
numeric_cols = df.select_dtypes(include='number').columns
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())
 
print("Missing values remaining:")
print(df.isnull().sum().sum())

Missing values remaining:
0


In [7]:
from sklearn.preprocessing import LabelEncoder


In [8]:
 
text_cols = df.select_dtypes(include='object').columns
print("Text columns:", list(text_cols))
 
le = LabelEncoder()
for col in text_cols:
    df[col] = le.fit_transform(df[col].astype(str))
 
print(df.shape)
df.head()

Text columns: ['CODE_GENDER', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'NAME_INCOME_TYPE', 'ORGANIZATION_TYPE']
(307511, 32)


,TARGET,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,CNT_CHILDREN,CODE_GENDER,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,FLAG_OWN_CAR,FLAG_OWN_REALTY,...,AGE_YEARS,EMPLOYMENT_YEARS,REGISTRATION_YEARS,EMPLOYMENT_TO_AGE,CREDIT_TO_AGE,INCOME_PER_PERSON,ANNUITY_TO_GOODS,LOANS_PER_YEAR,TOTAL_DEBT_TO_INCOME,PAST_DEFAULT_RATE
0,1,202500.0,406597.5,24700.5,0,1,4,3,0,1,...,25.9,1.7,10.0,0.065637,15698.745174,202500.0,0.070372,0.308880,0.242747,0.0
1,0,270000.0,1293502.5,35698.5,0,0,1,1,0,0,...,45.9,3.3,3.2,0.071895,28180.882353,270000.0,0.031606,0.087146,0.000000,0.0
2,0,67500.0,135000.0,6750.0,0,1,4,3,1,1,...,52.2,0.6,11.7,0.011494,2586.206897,67500.0,0.050000,0.038314,0.000000,0.0
3,0,135000.0,312682.5,29686.5,0,0,4,0,0,1,...,52.1,8.3,26.9,0.159309,6001.583493,135000.0,0.099955,0.103734,0.297350,0.0
4,0,121500.0,513000.0,21865.5,0,1,4,3,0,1,...,54.6,8.3,11.8,0.152015,9395.604396,121500.0,0.042623,0.018315,0.000000,0.0


In [9]:
df.to_csv('../data/updated_clean_data.csv', index=False)
